# MFFT Ablation Study
**Validates Tables 4 & 5: frequency components, fusion strategies, band count, and attention mechanisms.**

In [ ]:
# Cell 1: Imports & Setup
import os, sys, math, json, time, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm

# Robust PROJECT_ROOT detection (walk up from CWD)
_p = Path.cwd().resolve()
for __ in range(10):
    if (_p / 'AGENTS.md').exists() or (_p / '.git').exists():
        break
    _parent = _p.parent
    if _parent == _p:
        _p = _p / 'ai-image-detection-research'
        break
    _p = _parent
PROJECT_ROOT = _p
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'model'))
print(f'Project root: {PROJECT_ROOT}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Cell 2: Config
from src.dataset import AIDetectionDataset, ImageTransform
from src.config import Config

# Authoritative PROJECT_ROOT from module location
import src
PROJECT_ROOT = Path(src.__file__).resolve().parent.parent.parent
os.chdir(PROJECT_ROOT)

cfg = Config()
cfg.training.epochs = 20
cfg.training.model_variant = 'base'
cfg.training.image_size = 384
cfg.training.batch_size = 8
cfg.training.mixed_precision = False
cfg.training.num_workers = 0
cfg.training.gradient_accumulation_steps = 1
cfg.dataset.val_split = 0.1
cfg.dataset.test_split = 0.1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')

In [ ]:
# Cell 3: Load Dataset
full_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT),
    metadata_paths=cfg.dataset.metadata_paths,
    transform=None,
    is_train=True,
    size=cfg.training.image_size,
    undersample=True,
)

print(f'Total samples: {len(full_dataset)}')
print(f'Real: {sum(1 for _, l in full_dataset.samples if l==0)}')
print(f'AI:   {sum(1 for _, l in full_dataset.samples if l==1)}')

In [ ]:
# Cell 4: Train/Val/Test Split
from sklearn.model_selection import train_test_split

labels = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

train_idx, temp_idx = train_test_split(
    indices, test_size=cfg.dataset.val_split + cfg.dataset.test_split,
    stratify=labels, random_state=42,
)
temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=cfg.dataset.test_split / (cfg.dataset.val_split + cfg.dataset.test_split),
    stratify=temp_labels, random_state=42,
)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

train_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=True),
    is_train=True, size=cfg.training.image_size, undersample=False,
)
train_dataset.samples = [full_dataset.samples[i] for i in train_idx]

val_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=False),
    is_train=False, size=cfg.training.image_size, undersample=False,
)
val_dataset.samples = [full_dataset.samples[i] for i in val_idx]

test_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=False),
    is_train=False, size=cfg.training.image_size, undersample=False,
)
test_dataset.samples = [full_dataset.samples[i] for i in test_idx]

train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.training.batch_size, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.training.batch_size, shuffle=False, num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')

In [ ]:
# Cell 5: Ablation Configuration = Manual Table 4 & 5
from src.model import build_mfft, count_parameters

# Each ablation config maps to a row in Table 4 or Table 5
ABLATIONS = [
    # ── Table 4: Frequency Components ──
    {
        "name": "spatial_only",
        "label": "Spatial-only (no bands)",
        "table": 4,
        "ablation": {"spatial_only": True, "use_fga": False, "fusion_mode": "concat"},
    },
    {
        "name": "band_low",
        "label": "Low-frequency only",
        "table": 4,
        "ablation": {"skip_bands": ["mid", "high"], "use_fga": False, "fusion_mode": "concat"},
    },
    {
        "name": "band_mid",
        "label": "Mid-frequency only",
        "table": 4,
        "ablation": {"skip_bands": ["low", "high"], "use_fga": False, "fusion_mode": "concat"},
    },
    {
        "name": "band_high",
        "label": "High-frequency only",
        "table": 4,
        "ablation": {"skip_bands": ["low", "mid"], "use_fga": False, "fusion_mode": "concat"},
    },
    {
        "name": "fusion_concat",
        "label": "All bands (concat)",
        "table": 4,
        "ablation": {"fusion_mode": "concat", "use_fga": False},
    },
    {
        "name": "fusion_avg",
        "label": "All bands (avg)",
        "table": 4,
        "ablation": {"fusion_mode": "avg", "use_fga": False},
    },
    {
        "name": "fusion_max",
        "label": "All bands (max)",
        "table": 4,
        "ablation": {"fusion_mode": "max", "use_fga": False},
    },
    {
        "name": "no_fga",
        "label": "All bands (cross-attn, no FGA)",
        "table": 4,
        "ablation": {"fusion_mode": "attention", "use_fga": False},
    },
    # ── Table 5: Band Count ──
    {
        "name": "bands_2",
        "label": "2 bands (low, high)",
        "table": 5,
        "variant": "base",
        "ablation": {"skip_bands": ["mid"], "fusion_mode": "attention", "use_fga": True},
    },
    {
        "name": "bands_4",
        "label": "4 bands (low, low-mid, mid-high, high)",
        "table": 5,
        "variant": "large",
        "ablation": {"fusion_mode": "attention", "use_fga": True},
    },
]

print(f'{len(ABLATIONS)} ablation experiments defined:')
for a in ABLATIONS:
    print(f'  Table {a["table"]}: {a["label"]}')

In [ ]:
# Cell 6: Training Loop for a Single Ablation Config
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR

NUM_EPOCHS = 20
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.training.label_smoothing)

def train_ablation(ablation_cfg):
    name = ablation_cfg["name"]
    print(f'\n{"="*60}')
    print(f'Running: {ablation_cfg["label"]}')
    print(f'{"="*60}')

    variant = ablation_cfg.get('variant', 'base')
    model = build_mfft(variant, ablation=ablation_cfg['ablation'])
    model = model.to(device)
    params = count_parameters(model)
    print(f'Parameters: {params:,}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=min(500, len(train_loader)))
    cosine = CosineAnnealingWarmRestarts(optimizer, T_0=NUM_EPOCHS * len(train_loader), T_mult=2, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[min(500, len(train_loader))])

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc = 0.0

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        pbar = tqdm(train_loader, desc=f'{name} E{epoch+1}/{NUM_EPOCHS}')
        for images, labels in pbar:
            try:
                images, labels = images.to(device), labels.to(device)
                logits = model(images)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()
                total_loss += loss.item()
                preds = logits.argmax(dim=-1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
            except Exception as e:
                print(f'  Warning: skipping bad batch: {e}')
                optimizer.zero_grad()
                continue
            if (total % (cfg.training.batch_size * 10)) == 0 and total > 0:
                acc = correct / total * 100
                pbar.set_postfix({'loss': f'{total_loss/(total/cfg.training.batch_size):.4f}', 'acc': f'{acc:.2f}%'})

        train_acc = correct / total * 100
        history['train_acc'].append(train_acc)
        history['train_loss'].append(total_loss / len(train_loader))

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                try:
                    images, labels = images.to(device), labels.to(device)
                    logits = model(images)
                    loss = criterion(logits, labels)
                    val_loss += loss.item()
                    preds = logits.argmax(dim=-1)
                    val_correct += (preds == labels).sum().item()
                    val_total += labels.size(0)
                except Exception as e:
                    print(f'  Warning: bad val batch: {e}')
                    continue

        val_acc = val_correct / val_total * 100
        history['val_acc'].append(val_acc)
        history['val_loss'].append(val_loss / len(val_loader))
        print(f'  train={train_acc:.2f}%, val={val_acc:.2f}%')

        if val_acc > best_acc:
            best_acc = val_acc

    # Test evaluation
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            try:
                images, labels = images.to(device), labels.to(device)
                logits = model(images)
                preds = logits.argmax(dim=-1)
                test_correct += (preds == labels).sum().item()
                test_total += labels.size(0)
            except Exception as e:
                print(f'  Warning: bad test batch: {e}')
                continue

    test_acc = test_correct / test_total * 100
    print(f'  Best val: {best_acc:.2f}%, Test: {test_acc:.2f}%')

    result = {
        'name': name,
        'label': ablation_cfg['label'],
        'table': ablation_cfg['table'],
        'params': params,
        'final_train_acc': round(train_acc, 2),
        'best_val_acc': round(best_acc, 2),
        'test_acc': round(test_acc, 2),
        'history': {k: [round(v, 4) for v in vs] for k, vs in history.items()},
    }
    return result

In [ ]:
# Cell 7: Run All Ablations
RESULTS_DIR = PROJECT_ROOT / 'paper' / 'results' / 'ablation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

all_results = []
for ablation_cfg in ABLATIONS:
    result = train_ablation(ablation_cfg)
    all_results.append(result)

    # Save individual result
    with open(RESULTS_DIR / f'{result["name"]}.json', 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  Saved {RESULTS_DIR / result["name"]}.json')

print(f'\nAll {len(all_results)} ablations complete.')

In [ ]:
# Cell 8: Summary Table (mirrors manuscript Table 4 & 5)
print("=" * 72)
print(f"{'TABLE 4: Ablation of Frequency Components':^72}")
print("=" * 72)
print(f"{'Configuration':<36} {'Params':>8} {'Val Acc':>8} {'Test Acc':>8}")
print("-" * 72)
for r in all_results:
    if r['table'] == 4:
        print(f"{r['label']:<36} {r['params']:>8,} {r['best_val_acc']:>7.2f}% {r['test_acc']:>7.2f}%")

print("\n" + "=" * 72)
print(f"{'TABLE 5: Ablation of Number of Frequency Bands':^72}")
print("=" * 72)
print(f"{'Configuration':<36} {'Params':>8} {'Val Acc':>8} {'Test Acc':>8}")
print("-" * 72)
for r in all_results:
    if r['table'] == 5:
        print(f"{r['label']:<36} {r['params']:>8,} {r['best_val_acc']:>7.2f}% {r['test_acc']:>7.2f}%")

# Save summary CSV
import csv
with open(RESULTS_DIR / 'ablation_summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['Configuration', 'Table', 'Params', 'Best Val Acc (%)', 'Test Acc (%)'])
    for r in all_results:
        w.writerow([r['label'], r['table'], r['params'], r['best_val_acc'], r['test_acc']])

with open(RESULTS_DIR / 'ablation_summary.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'\nSummary saved to {RESULTS_DIR}/')

In [ ]:
# Cell 9: Comparison Plot
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Table 4 — Frequency Components
t4 = [r for r in all_results if r['table'] == 4]
labels = [r['label'] for r in t4]
vals = [r['best_val_acc'] for r in t4]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(labels)))
axes[0].barh(range(len(labels)), vals, color=colors)
axes[0].set_yticks(range(len(labels)))
axes[0].set_yticklabels(labels, fontsize=8)
axes[0].set_xlabel('Best Val Accuracy (%)')
axes[0].set_title('Ablation: Frequency Components (Table 4)')
axes[0].invert_yaxis()
for i, v in enumerate(vals):
    axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=8)

# Table 5 — Band Count
t5 = [r for r in all_results if r['table'] == 5]
labels5 = [r['label'] for r in t5]
vals5 = [r['best_val_acc'] for r in t5]
axes[1].bar(labels5, vals5, color=['#2196F3', '#4CAF50', '#FF9800'][:len(labels5)])
axes[1].set_ylabel('Best Val Accuracy (%)')
axes[1].set_title('Ablation: Band Count (Table 5)')
for i, v in enumerate(vals5):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {RESULTS_DIR / "ablation_summary.png"}')